In [ ]:
import numpy as np
import pandas as pd
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

# Metrics
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    average_precision_score,  # PR-AUC
    roc_auc_score
)

# Preprocessing + Model
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Load data - Using the unified master_df.csv
master_path = "../data/processed/master_df.csv"
if not os.path.exists(master_path):
    raise FileNotFoundError(f"Could not find {master_path}. Please run preprocessing_and_features.ipynb first.")

master_df = pd.read_csv(master_path)
master_df['DATE'] = pd.to_datetime(master_df['DATE'])

print("Master dataset shape:", master_df.shape)

label_col = "crash_tomorrow"

# temporal split: <=2023 for training, 2024+ for validation
train_mask = master_df['DATE'].dt.year <= 2023
test_mask  = master_df['DATE'].dt.year > 2023

train_df = master_df[train_mask].copy()
val_df = master_df[test_mask].copy()

print("Train_df shape:", train_df.shape)
print("Val_df shape:  ", val_df.shape)

drop_cols = [
    "geometry",
    "CountyNameTxt",
    "Region",
    "RdwyTypeCde_txtCARTO",
    "DATE"
]

categorical_cols = [
    "WeatherCde_txtCARTO",
    "SurfaceConditionCde_txtCARTO",
]

# Feature Engineering: Extract cell coordinates
for df_tmp in [train_df, val_df]:
    df_tmp[["cell_x", "cell_y"]] = df_tmp["cell_id"].str.split("_", expand=True).astype(int)

# Target and Features
y_train = train_df[label_col].astype(int).values
y_val = val_df[label_col].astype(int).values

X_train_df = train_df.drop(columns=[label_col] + drop_cols + ["cell_id"], errors="ignore")
X_val_df = val_df.drop(columns=[label_col] + drop_cols + ["cell_id"], errors="ignore")

numeric_cols = [c for c in X_train_df.columns if c not in categorical_cols]

print(f"X_train shape: {X_train_df.shape}")
print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")
print("Positive rate (train):", y_train.mean())
print("Positive rate (val):", y_val.mean())


FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/master_training.csv'

In [ ]:
# Class counts for imbalance handling
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()

print("Train positives:", n_pos)
print("Train negatives:", n_neg)

# Similar to logistic pos_weight: negatives / positives
pos_weight = n_neg / n_pos
class_weight = {0: 1.0, 1: float(pos_weight)}

print("Computed pos_weight (neg/pos):", pos_weight)
print("Using class_weight:", class_weight)


Train positives: 21788
Train negatives: 353108
Computed pos_weight (neg/pos): 16.206535707729024
Using class_weight: {0: 1.0, 1: 16.206535707729024}


In [ ]:
from sklearn.utils._testing import ignore_warnings
from sklearn.exceptions import ConvergenceWarning

# Define the preprocessor to handle numeric scaling and categorical encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

# Build a pipeline: Preprocessor -> MLPClassifier
# NOTE: MLPClassifier does not support class_weight directly. 
# For imbalance, you could use oversampling (SMOTE) or undersampling before training.
ann_pipeline = Pipeline([
    ("prep", preprocessor),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(64, 32),  # 2 hidden layers
        activation="relu",
        solver="adam",
        learning_rate_init=1e-3,
        max_iter=50, # Increased slightly
        alpha=1e-4,
        random_state=42
    ))
])

@ignore_warnings(category=ConvergenceWarning)
def fit_ann_model(pipe, X, y):
    return pipe.fit(X, y)

print("Training ANN (MLP) model...")
ann_model = fit_ann_model(ann_pipeline, X_train_df, y_train)
print("ANN model trained.")


ANN model trained.


In [ ]:
def hit_rate_at_k(y_true, y_prob, k):
    # Measures what fraction of the top-k highest-probability cells actually had a crash.
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    idx_sorted = np.argsort(y_prob)[::-1]
    top_idx = idx_sorted[:k]
    hits = y_true[top_idx].sum()
    return hits / float(k)


# Get probabilities on validation set using the DataFrame
y_val_proba = ann_model.predict_proba(X_val_df)[:, 1]

brier = brier_score_loss(y_val, y_val_proba)
logloss = log_loss(y_val, y_val_proba)
pr_auc = average_precision_score(y_val, y_val_proba)
roc_auc = roc_auc_score(y_val, y_val_proba)

# Choose a k for hit-rate
k = 500
hr_k = hit_rate_at_k(y_val, y_val_proba, k)

print("=== ANN (MLP) Model – Validation Metrics ===")
print(f"Brier score:        {brier:.6f}")
print(f"Log-loss:           {logloss:.6f}")
print(f"PR-AUC (avg prec):  {pr_auc:.6f}")
print(f"ROC-AUC:            {roc_auc:.6f}")
print(f"Hit-rate@k (k={k}): {hr_k:.6f}")


=== ANN (MLP) Model – Validation Metrics ===
Brier score:        0.049809
Log-loss:           0.198600
PR-AUC (avg prec):  0.112763
ROC-AUC:            0.689069
Hit-rate@k (k=500): 0.168000
